# ESM2 Masked Language Model

![ESM2 Masked Language Model](https://proto-bio.github.io/proto-assets/images/tool/esm2/hero.png)

This notebook demonstrates the four core operations of the ESM2 toolkit: extracting sequence embeddings, sampling mutations, scoring sequences, and computing gradients. ESM2 is a protein language model trained on millions of protein sequences using masked language modeling, and its learned representations capture structural and evolutionary relationships that support a range of downstream tasks.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("esm2")
display_overview("esm2")
display_docs_section("esm2", "Background")

## Available tools

In [ ]:
display_available_tools("esm2")

### `run_esm2_embeddings`

ESM2 produces dense vector representations for each input protein sequence. These mean-pooled embeddings compress the full sequence-length representation into a single fixed-dimension vector that captures structural, functional, and evolutionary properties. Embeddings from different sequences can be compared directly using cosine similarity or Euclidean distance, making them useful as features for machine learning models, for clustering related proteins, or for building sequence search indices.

In [ ]:
from proto_tools import (
    ESM2EmbeddingsConfig,
    ESM2EmbeddingsInput,
    run_esm2_embeddings,
)

In [ ]:
# Display docs
display_api_reference("esm2", "input", "run_esm2_embeddings")

# Two hemoglobin-like sequences to embed
sequences = [
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
    "MNLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
]
inputs = ESM2EmbeddingsInput(sequences=sequences)

In [ ]:
# Display config docs
display_api_reference("esm2", "config", "run_esm2_embeddings")

# Configure the 650M model | see docs above for all fields
config = ESM2EmbeddingsConfig(
    model_checkpoint="esm2_t33_650M_UR50D",
    batch_size=2,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the embeddings function
embeddings_result = run_esm2_embeddings(inputs, config)

In [ ]:
# Display output docs
display_api_reference("esm2", "output", "run_esm2_embeddings")

# Inspect the embedding dimension and first few values for each sequence
for i, emb in enumerate(embeddings_result.results):
    print(f"Sequence {i + 1}: embedding length = {len(emb.mean_embedding)}, first 5 values = {emb.mean_embedding[:5]}")

### `run_esm2_sample`

ESM2 sampling uses the model's predicted amino acid distributions to propose mutations at masked positions. There are two ways to specify which positions to sample:

1. **Custom masks** — directly mark positions with `_` in the input sequence to control exactly which residues are redesigned
2. **Masking strategy** — let the `MaskingStrategy` config automatically select positions using random, entropy-based, or max-logit methods

#### Approach 1: Custom masks with `_`

Use the `_` character to explicitly mark which positions should be redesigned. All other positions are held fixed. This is useful when you know exactly which residues you want to mutate — for example, redesigning a loop region or a specific binding interface.

In [ ]:
from proto_tools import (
    ESM2SampleConfig,
    ESM2SampleInput,
    run_esm2_sample,
)

# Display input docs
display_api_reference("esm2", "input", "run_esm2_sample")

# Mask positions 6-10 with _ to redesign that region
masked_input = ESM2SampleInput(sequences=["MVLSP_____VKAAW"])

In [ ]:
# Display config docs
display_api_reference("esm2", "config", "run_esm2_sample")

# No masking strategy needed — positions are already specified by _
config = ESM2SampleConfig(
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run sampling on the custom-masked input
custom_mask_result = run_esm2_sample(masked_input, config)

In [ ]:
# Display output docs
display_api_reference("esm2", "output", "run_esm2_sample")

# Show the original vs sampled — only the masked positions change
original = "MVLSPADKTNVKAAW"
for i, seq in enumerate(custom_mask_result.sequences):
    diffs = [j + 1 for j, (a, b) in enumerate(zip(original, seq)) if a != b]
    print(f"Original: {original}")
    print(f"Sampled:  {seq}  (mutated positions: {diffs})")

#### Approach 2: Automatic position selection with `MaskingStrategy`

Instead of manually placing `_` characters, you can let `MaskingStrategy` choose which positions to mask. The `method` parameter controls the selection logic:

- **`"random"`** (default) — uniform random selection
- **`"entropy"`** — targets positions where ESM2 is least certain about the original residue
- **`"max-logit"`** — targets positions where ESM2 most confidently predicts a different amino acid

Use `num_mutations` for an exact count or `mask_fraction` for a proportion of positions.

In [ ]:
from proto_tools.transforms.masking import MaskingStrategy

# Entropy-based masking to target the 5 most uncertain positions
strategy_input = ESM2SampleInput(sequences=["MVLSPADKTNVKAAW"])
strategy_config = ESM2SampleConfig(
    masking_strategy=MaskingStrategy(method="entropy", num_mutations=5),
    device="cuda",  # Change to "cpu" if no GPU available
)

strategy_result = run_esm2_sample(strategy_input, strategy_config)

In [ ]:
# The model automatically selected the 5 highest-entropy positions to mutate
original = "MVLSPADKTNVKAAW"
for i, seq in enumerate(strategy_result.sequences):
    diffs = [j + 1 for j, (a, b) in enumerate(zip(original, seq)) if a != b]
    print(f"Original: {original}")
    print(f"Sampled:  {seq}  (mutated positions: {diffs})")

### `run_esm2_score`

Pseudo-perplexity scoring masks each position in a sequence one at a time and measures how well the model predicts the original residue. Lower perplexity indicates that the sequence is more consistent with the evolutionary patterns the model learned during training, while higher perplexity suggests the sequence contains unusual or unlikely amino acid choices. This is useful for ranking designed sequences, filtering out poor candidates before expensive experimental validation, or identifying anomalous regions within a protein.

In [ ]:
from proto_tools import (
    ESM2ScoringConfig,
    ESM2ScoringInput,
    run_esm2_score,
)

In [ ]:
# Display docs
display_api_reference("esm2", "input", "run_esm2_score")

# Two short sequences to score
inputs = ESM2ScoringInput(sequences=["MKTAYIAKQR", "EVQLVESGGS"])

In [ ]:
# Display config docs
display_api_reference("esm2", "config", "run_esm2_score")

# Process 5 masked variants per forward pass | see docs above for all fields
config = ESM2ScoringConfig(
    batch_size=5,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the scoring function
score_result = run_esm2_score(inputs, config)

In [ ]:
# Display output docs
display_api_reference("esm2", "output", "run_esm2_score")

# Show all scoring metrics for each input sequence
for i, score in enumerate(score_result.scores):
    print(f"Sequence {i + 1}: {inputs.sequences[i]}")
    print(f"  Log-likelihood:     {score['log_likelihood']:.3f}")
    print(f"  Avg log-likelihood: {score['avg_log_likelihood']:.3f}")
    print(f"  Perplexity:         {score['perplexity']:.3f}")

### `run_esm2_gradient`

The gradient tool takes a relaxed `(L, 20)` distribution over amino acids and returns the gradient of ESM2's masked pseudo-log-likelihood objective with respect to those input logits. Each AA position is masked in turn, the model predicts it from the surrounding bidirectional context, and a per-chunk backward pass accumulates the gradient. Use this as a differentiable, structure-free pLM loss inside MCMC or gradient descent over relaxed protein sequences. Set `compute_gradient=False` to get the same scalar objective without the backward pass — useful for scoring proposals.

In [ ]:
from proto_tools import (
    ESM2GradientConfig,
    ESM2GradientInput,
    run_esm2_gradient,
)
from proto_tools.utils import one_hot_protein_logits


In [ ]:
# Display docs
display_api_reference("esm2", "input", "run_esm2_gradient")

# Seed a relaxed distribution from a discrete sequence (sharpness=2.0
# yields a biased-but-not-saturated softmax target).
logits = one_hot_protein_logits("MKTAYIAKQR", sharpness=2.0)
inputs = ESM2GradientInput(inputs=logits, temperature=0.6)


In [ ]:
# Display config docs
display_api_reference("esm2", "config", "run_esm2_gradient")

# Configure the 650M model | see docs above for all fields
config = ESM2GradientConfig(
    model_checkpoint="esm2_t33_650M_UR50D",
    hard=0.0,             # pure softmax; set hard=1.0 for a straight-through estimator
    compute_gradient=True,
    batch_size=8,         # AA positions per forward pass
    device="cuda",        # Change to "cpu" if no GPU available
)


In [ ]:
# Run the gradient function
gradient_result = run_esm2_gradient(inputs, config)


In [ ]:
# Display output docs
display_api_reference("esm2", "output", "run_esm2_gradient")

# One result per candidate; this call passed one, so results holds one
candidate = gradient_result.results[0]
term = candidate.terms[0]

print(f"Mean NLL:    {term.term_value:.3f}")
print(f"Perplexity:  {candidate.metrics['perplexity']:.3f}")
print(f"Avg log-likelihood: {candidate.metrics['avg_log_likelihood']:.3f}")
assert term.gradient is not None  # backward pass enabled
print(f"Gradient shape: ({len(term.gradient)}, {len(term.gradient[0])})")
print(f"Gradient row 0 (first 5 cols): {[round(v, 4) for v in term.gradient[0][:5]]}")


## Export Results

Results from each function can be exported to standard file formats for downstream analysis. Embeddings export to CSV with one row per sequence, sampled sequences export to FASTA for compatibility with other bioinformatics tools, and scores export to JSON with full metadata.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

embeddings_result.export("esm2_embeddings", export_path=output_dir, file_format="csv")
print(f"Embeddings exported to {output_dir / 'esm2_embeddings.csv'}")

strategy_result.export("esm2_sequences", export_path=output_dir, file_format="fasta")
print(f"Sampled sequences exported to {output_dir / 'esm2_sequences.fasta'}")

score_result.export("esm2_scores", export_path=output_dir, file_format="json")
print(f"Scores exported to {output_dir / 'esm2_scores.json'}")